In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
cd /content/drive/MyDrive

In [ ]:
ls

In [ ]:
import numpy as np
import os
import torch
from torch.utils.data import TensorDataset, DataLoader

class StandardScaler:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def transform(self, data):
        return (data - self.mean) / self.std

    def inverse_transform(self, data):
        return (data * self.std) + self.mean

def load_dataset(dataset_dir, batch_size, valid_batch_size=None, test_batch_size=None):
    data = {}
    for category in ["train", "val", "test"]:
        cat_data = np.load(os.path.join(dataset_dir, category + ".npz"))
        data["x_" + category] = cat_data["x"]
        data["y_" + category] = cat_data["y"]

    scaler = StandardScaler(
        mean=data["x_train"][..., 0].mean(), std=data["x_train"][..., 0].std()
    )
    for category in ["train", "val", "test"]:
        data["x_" + category][..., 0] = scaler.transform(data["x_" + category][..., 0])

    train_ds = TensorDataset(torch.Tensor(data["x_train"]), torch.Tensor(data["y_train"]))
    val_ds   = TensorDataset(torch.Tensor(data["x_val"]),   torch.Tensor(data["y_val"]))
    test_ds  = TensorDataset(torch.Tensor(data["x_test"]),  torch.Tensor(data["y_test"]))

    loader_kwargs = dict(num_workers=4, pin_memory=True, prefetch_factor=2, persistent_workers=True)
    data["train_loader"] = DataLoader(train_ds, batch_size=batch_size,      shuffle=True,  **loader_kwargs)
    data["val_loader"]   = DataLoader(val_ds,   batch_size=valid_batch_size, shuffle=False, **loader_kwargs)
    data["test_loader"]  = DataLoader(test_ds,  batch_size=test_batch_size,  shuffle=False, **loader_kwargs)

    data["scaler"] = scaler
    return data


In [ ]:
import torch
import torch.nn as nn
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

class TemporalEmbedding(nn.Module):
    def __init__(self, time, n_embd):
        super().__init__()
        self.day = time
        self.time_day = nn.Embedding(time, n_embd)
        self.time_week = nn.Embedding(7, n_embd)
        nn.init.xavier_uniform_(self.time_day.weight)
        nn.init.xavier_uniform_(self.time_week.weight)
    
    def forward(self, x): # x shape: (B, P, N, C)
        device = x.device 
        day_info = x[..., 1]  # (B, P, N)
        day_idx = (day_info[:, -1, :] * self.day).long().clamp(0, self.day - 1).to(device)
        day_embd = self.time_day(day_idx).transpose(1, 2).unsqueeze(-1)      # (B, n_embd, N, 1)

        week_info = x[..., 2]  # (B, P, N)
        week_idx = week_info[:, -1, :].long().clamp(0, 6).to(device)
        week_embd = self.time_week(week_idx).transpose(1, 2).unsqueeze(-1)    # (B, n_embd, N, 1)

        return day_embd + week_embd  # (B, n_embd, N, 1)
    
    
class SpatialEmbedding(nn.Module):
    def __init__(self, num_locations, n_embd):
        super().__init__()
        self.num_locations = num_locations
        self.location_embd = nn.Embedding(num_locations, n_embd)
        nn.init.xavier_uniform_(self.location_embd.weight)

    def forward(self, x):  # x shape: (B, P, N, C)
        B = x.shape[0]
        device = x.device 
        indices = torch.arange(self.num_locations, device=device)
        node_emb = self.location_embd(indices)
        node_emb = node_emb.unsqueeze(0).expand(B, -1, -1).transpose(1, 2).unsqueeze(-1)  # (B, n_embd, N, 1)
        return node_emb


class TokenEmbedding(nn.Module):
    def __init__(self, input_dim, input_len=12, n_embd=256):
        super().__init__()
        self.in_channels = input_dim * input_len
        self.tk_embd = nn.Conv2d(in_channels=self.in_channels, out_channels=n_embd, kernel_size=1)

    def forward(self, x): # x shape: (B, P, N, C)
        B, P, N, C = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, -1, N)  # (B, C*P, N)
        return self.tk_embd(x.unsqueeze(-1))           # (B, n_embd, N, 1)


class WeatherCrossAttention(nn.Module):
    """
    Traffic node features (query) attend to the weather time series (key/value).

    bias=False on weather projections guarantees: all-zero weather → keys/values
    are zero → cross-attention output is zero → residual leaves traffic features
    unchanged. Non-zero rainfall produces a learned additive correction.
    """
    def __init__(self, d_model=768, input_len=12, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads   = n_heads
        self.d_head    = d_model // n_heads
        self.input_len = input_len

        # bias=False: zero weather input → zero output (no spurious side-effect)
        self.w_proj_k = nn.Linear(1, d_model, bias=False)
        self.w_proj_v = nn.Linear(1, d_model, bias=False)
        self.q_proj   = nn.Linear(d_model, d_model)
        self.out_proj  = nn.Linear(d_model, d_model, bias=False)
        self.norm      = nn.LayerNorm(d_model)
        self.scale     = self.d_head ** -0.5

    def forward(self, traffic_feat, x):
        # traffic_feat: (B, N, d_model)
        # x:            (B, P, N, C) — weather is channel index 3
        B, N, D = traffic_feat.shape
        P = self.input_len
        H, Dh = self.n_heads, self.d_head

        weather = x[:, :, :, 3].permute(0, 2, 1)   # (B, N, P)
        BN = B * N
        w_flat = weather.reshape(BN, P, 1)           # (BN, P, 1)
        q_flat = traffic_feat.reshape(BN, 1, D)      # (BN, 1, D)

        k = self.w_proj_k(w_flat)  # (BN, P, D)
        v = self.w_proj_v(w_flat)  # (BN, P, D)
        q = self.q_proj(q_flat)    # (BN, 1, D)

        q = q.view(BN, 1, H, Dh).transpose(1, 2)    # (BN, H, 1, Dh)
        k = k.view(BN, P, H, Dh).transpose(1, 2)    # (BN, H, P, Dh)
        v = v.view(BN, P, H, Dh).transpose(1, 2)    # (BN, H, P, Dh)

        attn = torch.softmax(q @ k.transpose(-2, -1) * self.scale, dim=-1)  # (BN, H, 1, P)
        out  = (attn @ v).squeeze(2).reshape(BN, D)  # (BN, D)
        out  = self.out_proj(out).view(B, N, D)       # (B, N, D)

        return self.norm(traffic_feat + out)


class PFA(nn.Module):
    def __init__(self, gpt_layers=6, U=1): 
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained("gpt2")
        self.gpt2.h = self.gpt2.h[:gpt_layers]
        self.U = U

        for layer_idx, layer in enumerate(self.gpt2.h):
            for name, param in layer.named_parameters():
                if layer_idx < gpt_layers - self.U:
                    param.requires_grad = "ln" in name
                else:
                    param.requires_grad = "mlp" not in name

    def forward(self, x):
        return self.gpt2(inputs_embeds=x).last_hidden_state


class ST_LLM(nn.Module):
    def __init__(self, batch_size=32, num_nodes=266, input_len=12, output_len=12, gpt_layers=6, U=1):
        super().__init__()
        self.batch_size  = batch_size
        self.num_nodes   = num_nodes
        self.input_len   = input_len
        self.output_len  = output_len
        self.gpt_layers  = gpt_layers
        self.U           = U
        self.time        = 48
        self.n_embd      = 512
        self.gpt_embd    = 768

        # Traffic-only embeddings (channels 0-2: flow, day, week)
        self.time_embd    = TemporalEmbedding(self.time, self.n_embd)
        self.spatial_embd = SpatialEmbedding(self.num_nodes, self.n_embd)
        self.token_embd   = TokenEmbedding(input_dim=3, input_len=self.input_len, n_embd=self.n_embd)

        # Fuse time + spatial + token → gpt_embd
        self.feature_fuse = nn.Conv2d(in_channels=3*self.n_embd, out_channels=self.gpt_embd, kernel_size=1)

        # Weather cross-attention: conditions fused features on weather time series.
        # Zero rainfall → zero cross-attn output → residual preserves traffic features.
        self.weather_cross_attn = WeatherCrossAttention(d_model=self.gpt_embd, input_len=self.input_len)

        self.pfa              = PFA(gpt_layers=self.gpt_layers, U=self.U)
        self.regression_layer = nn.Conv2d(in_channels=self.gpt_embd, out_channels=self.output_len, kernel_size=1)

    def forward(self, history_data):    # x shape: (B, P, N, C)
        input_data = history_data
        B, P, N, C = input_data.shape

        # Spatial-temporal embeddings (traffic channels only, exclude weather ch=3)
        time_embd    = self.time_embd(input_data)                     # (B, n_embd, N, 1)
        spatial_embd = self.spatial_embd(input_data)                  # (B, n_embd, N, 1)
        token_embd   = self.token_embd(input_data[:, :, :, :3])      # (B, n_embd, N, 1)

        embd_feature = torch.cat([token_embd, time_embd, spatial_embd], dim=1)  # (B, 3*n_embd, N, 1)
        data_feature = self.feature_fuse(embd_feature)                # (B, gpt_embd, N, 1)
        data_feature = data_feature.squeeze(-1).permute(0, 2, 1)     # (B, N, gpt_embd)

        # Weather cross-attention: injects rainfall signal as conditional modulation
        data_feature = self.weather_cross_attn(data_feature, input_data)  # (B, N, gpt_embd)

        pfa_out = self.pfa(data_feature)                              # (B, N, gpt_embd)

        pfa_out = pfa_out.permute(0, 2, 1).unsqueeze(-1)             # (B, gpt_embd, N, 1)
        pred    = self.regression_layer(pfa_out).squeeze(-1)         # (B, output_len, N)
        
        return pred


In [ ]:
# train
import torch
import torch.nn as nn
import numpy as np
import time
import os
import logging
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

torch.manual_seed(313)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

# ── 超参数 ────────────────────────────────────────────────────────────────────
batch_size  = 32
lrate       = 5e-4
wdecay      = 1e-4
epochs      = 300
num_nodes   = 266
input_len   = 12
output_len  = 12
gpt_layers  = 6
U           = 1
dataset_dir = "w_texi_drop"
SAVE_DIR    = "./checkpoints"
SAVE_AFTER  = 20          # 20 轮以后才开始保存最优权重
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Logger ────────────────────────────────────────────────────────────────────
def get_logger(log_path):
    logger = logging.getLogger("stw_llm")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s  %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger

# ── 指标计算 ──────────────────────────────────────────────────────────────────
def calc_metrics(pred, real):
    mae  = np.mean(np.abs(pred - real))
    rmse = np.sqrt(np.mean((pred - real) ** 2))
    mask = real != 0
    mape = np.mean(np.abs((pred[mask] - real[mask]) / real[mask])) * 100
    wape = np.sum(np.abs(pred - real)) / np.sum(np.abs(real)) * 100
    return {"mae": mae, "rmse": rmse, "mape": mape, "wape": wape}

# ── Trainer ───────────────────────────────────────────────────────────────────
class trainer:
    def __init__(self, scaler, batch_size, learning_rate, wdecay,
                 num_nodes, input_len, output_len, gpt_layers, U):
        self.scaler     = scaler
        self.amp_scaler = GradScaler()
        self.model = ST_LLM(batch_size=batch_size, num_nodes=num_nodes,
                            input_len=input_len, output_len=output_len,
                            gpt_layers=gpt_layers, U=U).to(device)
        self.model = torch.compile(self.model)   # PyTorch 2.0+ 图优化
        self.optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=learning_rate, weight_decay=wdecay)
        self.criterion = nn.MSELoss()

    def train_loop(self, x, y):
        self.model.train()
        self.optimizer.zero_grad(set_to_none=True)
        with autocast():
            output = self.model(x)
            pred   = self.scaler.inverse_transform(output).unsqueeze(-1)
            loss   = self.criterion(pred, y)
        self.amp_scaler.scale(loss).backward()
        self.amp_scaler.step(self.optimizer)
        self.amp_scaler.update()
        return loss.item()

    def eval_loop(self, x, y):
        self.model.eval()
        with torch.no_grad(), autocast():
            output = self.model(x)
            pred   = self.scaler.inverse_transform(output).unsqueeze(-1)
            loss   = self.criterion(pred, y)
        return loss.item(), pred, y

# ── 评估一整个 loader ─────────────────────────────────────────────────────────
def evaluate(engine, loader):
    mse_list, preds, reals = [], [], []
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        mse, p, r = engine.eval_loop(x_batch, y_batch)
        mse_list.append(mse)
        preds.append(p)
        reals.append(r)
    pred_all = torch.cat(preds).cpu().numpy()   # 一次性转换，减少 CPU 开销
    real_all = torch.cat(reals).cpu().numpy()
    metrics  = calc_metrics(pred_all, real_all)
    metrics["mse"] = np.mean(mse_list)
    return metrics

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    log_path  = os.path.join(SAVE_DIR, "train.log")
    ckpt_path = os.path.join(SAVE_DIR, "best_model.pt")
    logger    = get_logger(log_path)

    dataloader = load_dataset(dataset_dir, batch_size=batch_size,
                              valid_batch_size=batch_size, test_batch_size=batch_size)
    engine = trainer(scaler=dataloader["scaler"], batch_size=batch_size,
                     learning_rate=lrate, wdecay=wdecay,
                     num_nodes=num_nodes, input_len=input_len, output_len=output_len,
                     gpt_layers=gpt_layers, U=U)

    best_val_mae = float("inf")
    logger.info(f"Training started  |  epochs={epochs}  lr={lrate}  wd={wdecay}")
    logger.info(f"Checkpoint dir: {SAVE_DIR}  |  save_after={SAVE_AFTER}")
    logger.info(f"{'Epoch':>6}  {'TrainMSE':>10}  {'MAE':>8}  {'RMSE':>8}  {'MAPE%':>8}  {'WAPE%':>8}  {'Time':>7}  Note")

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # --- train ---
        train_losses = []
        train_bar = tqdm(dataloader["train_loader"],
                         desc=f"Epoch {epoch:>3}/{epochs} [train]",
                         leave=False, dynamic_ncols=True)
        for x_batch, y_batch in train_bar:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            loss = engine.train_loop(x_batch, y_batch)
            train_losses.append(loss)
            train_bar.set_postfix(loss=f"{loss:.4f}")

        # --- val ---
        val_m      = evaluate(engine, dataloader["val_loader"])
        epoch_time = time.time() - t0

        # --- 20 轮以后保存最优权重（以 MAE 为准）---
        tag = ""
        if epoch > SAVE_AFTER and val_m["mae"] < best_val_mae:
            best_val_mae = val_m["mae"]
            torch.save(engine.model.state_dict(), ckpt_path)
            tag = "★ best"

        logger.info(
            f"{epoch:>6}/{epochs}  "
            f"{np.mean(train_losses):>10.4f}  "
            f"{val_m['mae']:>8.4f}  "
            f"{val_m['rmse']:>8.4f}  "
            f"{val_m['mape']:>8.2f}  "
            f"{val_m['wape']:>8.2f}  "
            f"{epoch_time:>6.1f}s  {tag}"
        )

    # ── 最终 Test 评估 ────────────────────────────────────────────────────────
    logger.info("=" * 70)
    logger.info(f"Loading best checkpoint  (best val MAE={best_val_mae:.4f})")
    engine.model.load_state_dict(torch.load(ckpt_path, map_location=device))

    test_m = evaluate(engine, dataloader["test_loader"])
    logger.info(
        f"[TEST]  MAE={test_m['mae']:.4f}  RMSE={test_m['rmse']:.4f}  "
        f"MAPE={test_m['mape']:.2f}%  WAPE={test_m['wape']:.2f}%  MSE={test_m['mse']:.4f}"
    )
    logger.info("=" * 70)

if __name__ == "__main__":
    main()
